# Content-Based & LLM-Based Recommendation System

# Phần 1: Content-Based Recommendation

In [3]:
# Import các thư viện cần thiết
import pandas as pd
import numpy as np
import re
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [4]:
# Load dataset và xem thông tin tổng quan
df = pd.read_csv('../../data/all_recipes_final.csv')
print(f"Dataset shape: {df.shape}")
print(f"Phân bố theo nguồn:")
print(df['source'].value_counts())
df.head()

Dataset shape: (10263, 12)
Phân bố theo nguồn:
source
dienmayxanh    8993
vnexpress       737
vncooking       533
Name: count, dtype: int64


,title,type_of_food,link,description,ingredients,step,note,num_of_ingredients,cook_time,num_of_people,calories,source
0,Cách muối dưa hành truyền thống,Món Tết,https://vnexpress.net/doi-song-cooking-cach-mu...,Dưa hành muối là món ăn truyền thống ngày Tết ...,"['1 kg hành củ tươi', 'Tro bếp hoặc nước vo gọ...",['Bước 1: Chọn hành củ: Nên chọn hành củ ta bá...,[],5,45 phút,8-10 người,459 kcal,vnexpress
1,Su hào xào mực - món cổ Tết Bát Tràng,Món Tết,https://vnexpress.net/doi-song-cooking-su-hao-...,Đĩa xào khô ráo với su hào giòn ngọt quyện với...,"['2 củ su hào non', '1 con mực khô', '1/2 củ c...",['Bước 1: Chọn và sơ chế mực: Người dân làng g...,['Su hào xào mực cùng với canh măng mực là hai...,6,50 phút,4 - 5 người,1.162 kcal,vnexpress
2,Canh măng ngày Tết cổ truyền Hà Nội,Món Tết,https://vnexpress.net/doi-song-cooking-canh-ma...,"Măng ngấu vị, giòn ngon, móng giò hầm vừa độ s...","['800 gr măng khô', '2 móng giò lợn', 'Nước dù...","['Bước 1: Chọn măng khô: Theo lối cũ, người nộ...",['Nếu tận dụng nước luộc gà nấu canh măng thì ...,6,100 phút,8 - 10 người,4.930 kcal,vnexpress
3,Giả hạnh nhân - món ngon Tết xưa Hà Nội,Món Tết,https://vnexpress.net/doi-song-cooking-gia-han...,Đây là món ăn cổ truyền thường thấy trong cỗ T...,"['2 bộ lòng mề gà', '100 gr lạc', '50 gr hạt đ...",['Bước 1: Chọn và sơ chế lạc: Chọn lạc khô chắ...,['Hạnh nhân xào (hay giả hạnh nhân) là món ăn ...,8,60 phút,4-5 người,1.112 kcal,vnexpress
4,Chả bì ớt xiêm xanh,Món Tết,https://vnexpress.net/doi-song-cooking-cha-bi-...,"Chả bì bóng đẹp, gói đều tay. Khi ăn vị ngọt m...","['500 gr giò sống', '300 gr bì lợn', '20 - 30 ...","['Bước 1: Chọn và sơ chế bì lợn, chuẩn bị giò ...",['Nên sơ chế kỹ bì lợn để chả được thơm. Tùy t...,6,60 phút,5-6 người,2.512 kcal,vnexpress


## 1.1. Data Preprocessing

In [5]:
# Xử lý missing values
data = df.copy()

data['title'] = data['title'].fillna('')
data['description'] = data['description'].fillna('')
data['step'] = data['step'].fillna('[]')
data['ingredients'] = data['ingredients'].fillna('[]')
data['type_of_food'] = data['type_of_food'].fillna('Unknown')

print("Missing values after handling:")
print(data[['title', 'description', 'step', 'ingredients', 'type_of_food', 'calories', 'cook_time']].isnull().sum())

Missing values after handling:
title              0
description        0
step               0
ingredients        0
type_of_food       0
calories        9826
cook_time        295
dtype: int64


In [6]:
# Xử lý missing values
data = df.copy()

data['title'] = data['title'].fillna('')
data['description'] = data['description'].fillna('')
data['step'] = data['step'].fillna('[]')
data['ingredients'] = data['ingredients'].fillna('[]')
data['type_of_food'] = data['type_of_food'].fillna('Unknown')

print("Missing values after handling:")
print(data[['title', 'description', 'step', 'ingredients', 'type_of_food', 'calories', 'cook_time']].isnull().sum())

Missing values after handling:
title              0
description        0
step               0
ingredients        0
type_of_food       0
calories        9826
cook_time        295
dtype: int64


In [7]:
# Tạo TF-IDF matrix từ text features
data['combined_text'] = data['title_clean'] + ' ' + data['description_clean'] + ' ' + data['step_clean']

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=1,
    max_df=0.95
)

tfidf_matrix = tfidf_vectorizer.fit_transform(data['combined_text'])
print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")

KeyError: 'title_clean'

In [ ]:
# Hàm lấy recommendations dựa trên Ingredient TF-IDF
def get_ingredient_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n=5):
    sim_scores = list(enumerate(similarity_matrix[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['ing_tfidf_score'] = scores
    
    return result

def recommend_by_title_ingredient_tfidf(title, df, similarity_matrix, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\nInput: {df.loc[recipe_idx, 'title']}")
    print(f"   Ingredients: {df.loc[recipe_idx, 'ingredients_text'][:100]}...")
    print("\nIngredient TF-IDF Recommendations:")
    
    return get_ingredient_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n)

print("Ingredient TF-IDF recommendation functions ready!")

In [ ]:
# Hàm lấy recommendations dựa trên hybrid approach
def get_hybrid_recommendations(recipe_idx, hybrid_sim, tfidf_sim, ing_tfidf_sim, df, top_n=5):
    sim_scores = list(enumerate(hybrid_sim[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    hybrid_scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['hybrid_score'] = hybrid_scores
    result['tfidf_score'] = [tfidf_sim[recipe_idx][i] for i in recipe_indices]
    result['ing_tfidf_score'] = [ing_tfidf_sim[recipe_idx][i] for i in recipe_indices]
    
    return result

def recommend_by_title_hybrid(title, df, hybrid_sim, tfidf_sim, ing_tfidf_sim, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"❌ No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\nInput: {df.loc[recipe_idx, 'title']}")
    print(f"   Type: {df.loc[recipe_idx, 'type_of_food']}, Calories: {df.loc[recipe_idx, 'calories']}, Time: {df.loc[recipe_idx, 'cook_time']}")
    print("\nHybrid Recommendations:")
    
    return get_hybrid_recommendations(recipe_idx, hybrid_sim, tfidf_sim, ing_tfidf_sim, df, top_n)

In [ ]:
# Tính Keyword-based similarity matrix
keywords_sets = data['keywords'].tolist()
keyword_similarity_matrix = compute_jaccard_similarity_matrix(keywords_sets)
print(f"Keyword Similarity Matrix shape: {keyword_similarity_matrix.shape}")

## 1.7. Testing All Methods - Comprehensive Comparison

## 2.1. Chuẩn bị data

In [ ]:
# SENTENCE EMBEDDINGS ( TF-IDF )

stopwords={
    "là", "của", "và", "những", "các", "cho", "với", "trong", "để", "khi",
    "một", "có", "được", "từ", "như", "người", "bạn", "hãy", "sẽ", "đã", "đang",
    "thì", "mà", "bị", "bởi", "cả", "lại", "nên", "này", "kia", "làm", "rằng",
    "về", "do", "bằng", "phải", "tại", "theo", "ra", "vào", "lên", "xuống",
    "đến", "qua", "bởi_vì", "nếu", "nhưng", "tuy_nhiên", "mặc_dù", "hoặc", "hay",
    "rất", "quá", "lắm", "nhiều", "ít", "hơn", "nhất", "khá", "chỉ", "mỗi", "từng",
    "không", "chưa", "chẳng", "đừng", "chớ", "vẫn", "cũng", "thôi", "nhé", "nha","muỗng", "thìa", "gam", "gram", "kg", "kilogam", "lít", "ml", "chén", "bát",
    "tô", "dĩa", "đĩa", "trái", "quả", "củ", "nhánh", "tép", "lát", "khứa", "miếng",
    "ổ", "ổ_bánh_mì", "lon", "hộp", "gói", "bao", "giọt", "nhúm", "nắm",
    "bước", "cách", "làm", "thực_hiện", "chuẩn_bị", "sơ_chế", "chế_biến",
    "hướng_dẫn", "thành_phẩm", "lưu_ý", "mẹo", "bí_quyết", "thưởng_thức",
    "bắt_đầu", "tiếp_theo", "sau_đó", "cuối_cùng", "hoàn_thành",
    "ngon", "vị", "hương_vị", "món", "ăn", "thơm", "hấp_dẫn", "đậm_đà",
    "gia_vị", "nêm", "nếm", "vừa_ăn", "khẩu_vị", "gia_đình", "bếp",
    "khoảng", "độ", "phút", "giờ", "nóng", "lạnh", "nguội"
}

# Viết hàm tiền xử lý dữ liệu
def processing_data(text):
  text = str(text)
  # Chuyển về từ thường
  text = text.lower()
  # Xóa dấu câu, ký tự đặc biệt
  text = re.sub(r'[^\w\s]', ' ', text)
  # Tách từ bằng underthesea
  text = word_tokenize(text, format="text")
  # Xóa khoảng trắng
  text = re.sub(r'\s+', ' ', text).strip()
  # Lọc stopword
  words = text.split()
  valid_words = []
  for word in words:
      # Kiểm tra từ đó (hoặc từ gốc thay thế dấu_) có trong stopword không
      if word not in stopwords and word.replace('_', ' ') not in stopwords:
          valid_words.append(word)
  return ' '.join(valid_words)

# Xử lý dữ liệu
print("Đang tiền xử lý dữ liệu văn bản...")
tqdm.pandas()
data_llm['embed_tf'] = data_llm['combined_text'].progress_apply(processing_data)

# Sử dụng TfidfVectorizer để chuyển cột combined_text của các món ăn về tf-idf
vectorizer = TfidfVectorizer()
overview_matrix = vectorizer.fit_transform(data_llm['embed_tf'])

# Tính toán cosine bằng linear_kernel
cosine_sim = linear_kernel(overview_matrix, overview_matrix)

# Đánh index cho các món ăn bằng pd.Series() và lưu trong biến mapping
mapping = pd.Series(data_llm.index, index=data_llm['title']).drop_duplicates()

print("Đang chuyển đổi sang định dạng FAISS...")
# Chuyển Sparse Matrix -> Dense Matrix -> Float32
tfidf_dense = overview_matrix.toarray().astype('float32')

# Khởi tạo Index FAISS
d = tfidf_dense.shape[1] # Số chiều = Số lượng từ vựng (Vocabulary size)

# Dùng IndexFlatIP
index = faiss.IndexFlatIP(d)

# Thêm dữ liệu vào Index
index.add(tfidf_dense)
print(f"Đã thêm {index.ntotal} món ăn vào FAISS Index.")

#  Lưu FAISS Index (Chứa cấu trúc tìm kiếm) và Mapping (Chứa quan hệ Tên món -> ID)
faiss.write_index(index, "food_tfidf.index")

with open("food_mapping.pkl", "wb") as f:
    pickle.dump(mapping, f)

print("Đã lưu Index và Mapping xuống ổ cứng!")

In [ ]:
if __name__ == "__main__":
    # Chọn model : tf-idf hoặc Sbert
    using_model= 'tf-idf'

    query = input()
    print(f"\n--- Những món ăn liên quan phù hợp với yêu cầu '{query}' ---")

    # Lấy ứng viên
    if using_model == 'Sbert':
        candidates = search_recipes_sbert(query, top_n=50)
    else:
        candidates = search_recipes_tf_idf(query, top_n=50)
    display(candidates[['title','description', 'link']].head(10))

    # LLM Reranking
    print(f"\n--- Kết quả rerank sau khi dùng LLM (gemini)  ---")
    reranked_results = llm_rerank(query, candidates, top_n=10)
    display(reranked_results[['title', 'description', 'link']])

In [ ]:
# LLM-BASED RERANKING (GEMINI)
from dotenv import load_dotenv
load_dotenv()

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")


genai.configure(api_key=GOOGLE_API_KEY)

def llm_rerank(user_query, candidate_df, top_n=5):

    # 1. Chuẩn bị dữ liệu Input
    candidates_json = candidate_df[['title', 'description']].to_json(orient='index', force_ascii=False)

    # 2. Cấu hình System Instruction (Chỉ thị cho AI)
    system_instruction = """
    You are a culinary expert assistant specializing in Vietnamese cuisine.
    Your task is to rerank the provided list of recipes based on their relevance to the user's query.

    CRITICAL OUTPUT RULES:
    1. Return ONLY a valid JSON object.
    2. Do not include any explanations, Markdown formatting (like ```json), or extra text.
    3. The JSON must follow this format: {"ranked_indices": [index1, index2, index3...]}
    4. Sort the indices from most relevant to least relevant.
    """

    # 3. Khởi tạo Model
    model = genai.GenerativeModel(
        model_name='gemini-2.5-flash-lite',
        system_instruction=system_instruction,
        generation_config={"response_mime_type": "application/json"} # Ép kiểu trả về là JSON
    )

    # 4. Tạo Prompt
    user_prompt = f"""
    User Query: "{user_query}"

    Candidate Recipes (JSON format with ID as key):
    {candidates_json}

    Please analyze and rank the top {top_n} most relevant recipes.
    """

    try:
        # 5. Gọi Gemini API
        response = model.generate_content(user_prompt)

        content = response.text.strip()

        # 6. Vệ sinh dữ liệu (Gemini đôi khi vẫn thêm markdown dù đã nhắc)
        if content.startswith("```json"):
            content = content.replace("```json", "").replace("```", "")
        elif content.startswith("```"):
            content = content.replace("```", "")

        content = content.strip()

        # 7. Parse kết quả
        match = re.search(r'(\{.*\}|\[.*\])', content, re.DOTALL)

        if match:
            clean_json_str = match.group(0) # Chỉ lấy phần JSON hợp lệ
            result_json = json.loads(clean_json_str)
        else:
            # Fallback nếu không tìm thấy pattern
            # Cố gắng clean thủ công
            content = content.replace("```json", "").replace("```", "").strip()
            result_json = json.loads(content)

        # Xử lý các trường hợp key khác nhau
        if isinstance(result_json, list):
            ranked_indices = result_json
        elif "ranked_indices" in result_json:
            ranked_indices = result_json["ranked_indices"]
        else:
            # Fallback: Lấy value đầu tiên nếu format lạ
            ranked_indices = list(result_json.values())[0]

        # 8. Trả về DataFrame đã sắp xếp
        # Chỉ lấy các index có tồn tại trong candidate_df để tránh lỗi
        valid_indices = [int(idx) for idx in ranked_indices if int(idx) in candidate_df.index]

        # Nếu LLM trả về ít hơn top_n hoặc rỗng, fallback về danh sách gốc
        if not valid_indices:
            return candidate_df.head(top_n)

        return data_llm.loc[valid_indices]

    except Exception as e:
        print(f"Error during Gemini reranking: {e}")
        # Fallback về danh sách gốc từ Semantic Search nếu lỗi
        return candidate_df.head(top_n)

## 2.4. Rerank bằng LLM (Gemini)

In [ ]:
# Các hàm gợi ý của sbert
# Load Model để mã hóa query mới của user
model = SentenceTransformer('keepitreal/vietnamese-sbert')

# Load FAISS Index
index_faiss = faiss.read_index("food_sbert.index")

# Load Embeddings gốc
embeddings = np.load("food_embeddings.npy")

# Tìm theo query của User - Bước đệm cho Reranking
def search_recipes_sbert(user_query, top_n=50):
    """
    Dùng SBERT để tìm top-N candidates phù hợp nhất với query
    """
    query_vec = model.encode([user_query])
    # Chuẩn hóa query
    faiss.normalize_L2(query_vec)
    # Tìm kiếm
    D, I = index_faiss.search(query_vec, k=top_n)

    # Lấy danh sách index
    result_indices = I[0]

    return data_llm.iloc[result_indices].copy()

In [ ]:
# Các hàm gợi ý của tf-idf
# Load data faiss tf-idf
index_tf_idf = faiss.read_index("food_tfidf.index")
with open("food_mapping.pkl", "rb") as f:
    mapping = pickle.load(f)

def search_recipes_tf_idf(user_query, top_n=50):
    """
    Tìm kiếm món ăn dựa trên query người dùng sử dụng TF-IDF.
    """
    # Tiền xử lý Query
    processed_query = processing_data(user_query)

    # Vector hóa Query
    query_sparse_matrix = vectorizer.transform([processed_query])

    # Chuyển đổi định dạng cho FAISS
    query_vec = query_sparse_matrix.toarray().astype('float32')

    D, I = index_tf_idf.search(query_vec, k=top_n)
    result_indices = I[0]

    # Lọc bỏ các giá trị -1 (nếu FAISS không tìm đủ k kết quả - hiếm gặp nhưng an toàn)
    result_indices = result_indices[result_indices != -1]

    return data_llm.iloc[result_indices].copy()

## 2.3. Các hàm đo độ tương đồng

In [ ]:
#  SENTENCE EMBEDDINGS ( SBERT )

# Load model keepitreal/vietnamese-sbert
print("Loading Embedding Model...")
model = SentenceTransformer('keepitreal/vietnamese-sbert')

# Generate embeddings
print("Encoding dataset...")
embeddings = model.encode(data_llm['combined_text'].tolist(), show_progress_bar=True)

# Chuẩn hóa L2
faiss.normalize_L2(embeddings)

# Khởi tạo Index FAISS
d = embeddings.shape[1]
index = faiss.IndexFlatIP(d)

# Thêm vector vào Index
index.add(embeddings)

# Lưu index và mảng Embeddings
faiss.write_index(index, "food_sbert.index")
np.save("food_embeddings.npy", embeddings)
print("Đã lưu xong: 'food_sbert.index' và 'food_embeddings.npy'")

## 2.2. Embedding dữ liệu theo 2 cách: TF-IDF và SBERT

In [ ]:
data_llm.head()

In [ ]:
# Tạo combined text: Title + Description
data_llm['combined_text'] = data_llm['title'] + '. ' + data_llm['description']

print(f"Loaded {len(data_llm)} recipes.")

In [ ]:
# SETUP & DATA LOADING

csv_file = '../../data/all_recipes_final.csv'
data_llm = pd.read_csv(csv_file)

In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import google.generativeai as genai
import os
import json
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel
from underthesea import word_tokenize
from tqdm import tqdm
import faiss
import pickle

# Phần 2: LLM-Based Recommendation

## 1.8. Content-Based Summary

### 4 Phương pháp Content-Based Recommendation đã implement:

1. **TF-IDF Based**: Text similarity (title + description + steps) với Cosine similarity
   - Ưu: Tốt cho text matching, phát hiện món ăn có cách nấu tương tự
   - Nhược: Phụ thuộc vào chất lượng text
   - Use case: Tìm món có description/cách nấu giống nhau

2. **Ingredient TF-IDF** **RECOMMENDED**: TF-IDF trên ingredient text
   - Ưu: Xử lý variations trong ingredient naming, gán trọng số cho ingredients
   - Ưu: Không bị exact string matching problem
   - **Best choice cho ingredient-based recommendation**
   - Use case: Tìm món có nguyên liệu tương đồng

3. **Keyword/Tag-Based**: Extract keywords từ title (thịt bò, xào, canh...) và Jaccard similarity
   - Ưu: Đơn giản, focus vào main ingredients và cooking methods
   - Ưu: Dễ hiểu và giải thích, nhanh
   - Use case: Quick matching based on key terms

4. **Hybrid**: Weighted combination - TF-IDF (0.4) + Ingredient TF-IDF (0.6)
   - Ưu: Kết hợp text và ingredient signals, robust
   - Ưu: Cân bằng giữa text similarity và ingredient similarity
   - Use case: Best overall personalized recommendation

### So Sánh và Khuyến Nghị:

| Method | Độ chính xác | Speed | Complexity | Best For |
|--------|--------------|-------|------------|----------|
| TF-IDF | 3/5 | Trung bình | Đơn giản | Text/description matching |
| **Ing TF-IDF** | 4.5/5 | Trung bình | Đơn giản | **Best ingredient matching** |
| Keyword | 3/5 | Nhanh | Đơn giản | Quick keyword matching |
| Hybrid | 4.5/5 | Trung bình | Trung bình | Overall best |

### Top 3 Methods Recommended:
1. **Ingredient TF-IDF** - Best cho ingredient-based recommendation
2. **Hybrid** - Best overall khi kết hợp text + ingredient signals
3. **Keyword** - Nhanh nhất, tốt cho quick search

In [ ]:
# So sánh TẤT CẢ phương pháp với cùng 1 món
test_recipe = "gà kho"

print("="*100)
print("TF-IDF METHOD (Text-based)")
print("="*100)
display(recommender.recommend(test_recipe, method='tfidf', top_n=5))

print("\n" + "="*100)
print("INGREDIENT TF-IDF METHOD (Better Ingredient Matching)")
print("="*100)
display(recommender.recommend(test_recipe, method='ing_tfidf', top_n=5))

print("\n" + "="*100)
print("KEYWORD-BASED METHOD")
print("="*100)
display(recommender.recommend(test_recipe, method='keyword', top_n=5))

print("\n" + "="*100)
print("HYBRID METHOD (TF-IDF + Ingredient TF-IDF)")
print("="*100)
display(recommender.recommend(test_recipe, method='hybrid', top_n=5))

In [ ]:
# Khởi tạo recommender instance (4 methods: TF-IDF, Ing TF-IDF, Keyword, Hybrid)
print("Initializing Food Recommender with 4 methods...")
recommender = FoodRecommender(df)

In [ ]:
# Class FoodRecommender hoàn chỉnh - với Ingredient TF-IDF và Keyword-based
class FoodRecommender:
    
    def __init__(self, df):
        self.df = df.copy()
        self._preprocess()
        self._build_similarity_matrices()
    
    def _preprocess(self):
        self.df['ingredients_list'] = self.df['ingredients'].apply(parse_list_string)
        self.df['step_list'] = self.df['step'].apply(parse_list_string)
        
        self.df['title_clean'] = self.df['title'].apply(clean_text)
        self.df['description_clean'] = self.df['description'].fillna('').apply(clean_text)
        self.df['step_clean'] = self.df['step_list'].apply(lambda x: ' '.join([clean_text(s) for s in x]))
        self.df['combined_text'] = self.df['title_clean'] + ' ' + self.df['description_clean'] + ' ' + self.df['step_clean']
        
        self.df['cook_time_minutes'] = self.df['cook_time'].apply(parse_cook_time)
        self.df['calories_numeric'] = self.df['calories'].apply(parse_calories)
        
        self.df['ingredients_clean'] = self.df['ingredients_list'].apply(
            lambda x: set([clean_text(ing) for ing in x if ing])
        )
        
        # Ingredient text for TF-IDF
        self.df['ingredients_text'] = self.df['ingredients_list'].apply(
            lambda x: ' '.join([clean_text(ing) for ing in x if ing])
        )
        
        # Extract keywords for keyword-based recommendation
        self.df['keywords'] = self.df['title'].apply(extract_keywords)
        
        print("Data preprocessing completed!")
    
    def _build_similarity_matrices(self):
        print("Building TF-IDF similarity...")
        tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=1, max_df=0.95)
        tfidf_matrix = tfidf_vectorizer.fit_transform(self.df['combined_text'])
        self.tfidf_sim = cosine_similarity(tfidf_matrix)
        
        print("Building Ingredient TF-IDF similarity...")
        ing_tfidf_vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), min_df=2, max_df=0.8)
        ing_tfidf_matrix = ing_tfidf_vectorizer.fit_transform(self.df['ingredients_text'])
        self.ing_tfidf_sim = cosine_similarity(ing_tfidf_matrix)
        
        print("Building Keyword similarity...")
        # Reuse Jaccard function but for keywords only
        def jaccard_similarity(set1, set2):
            if len(set1) == 0 and len(set2) == 0:
                return 0.0
            intersection = len(set1.intersection(set2))
            union = len(set1.union(set2))
            return intersection / union if union > 0 else 0.0
        
        def compute_jaccard_similarity_matrix(items_list):
            n = len(items_list)
            similarity_matrix = np.zeros((n, n))
            
            for i in range(n):
                for j in range(i, n):
                    sim = jaccard_similarity(items_list[i], items_list[j])
                    similarity_matrix[i][j] = sim
                    similarity_matrix[j][i] = sim
            
            return similarity_matrix
        
        self.keyword_sim = compute_jaccard_similarity_matrix(self.df['keywords'].tolist())
        
        print("Building Hybrid similarity...")
        self.hybrid_sim = compute_hybrid_similarity(
            self.tfidf_sim, self.ing_tfidf_sim,
            w_tfidf=0.4, w_ing_tfidf=0.6
        )
        
        print("All similarity matrices ready!")
    
    def recommend(self, title, method='hybrid', top_n=5):
        """
        Recommend recipes based on title
        
        Args:
            method: 'tfidf', 'ing_tfidf', 'keyword', 'hybrid'
            top_n: Number of recommendations
        """
        matches = self.df[self.df['title'].str.contains(title, case=False, na=False)]
        if len(matches) == 0:
            print(f"No recipe found with title containing: {title}")
            return None
        
        recipe_idx = matches.index[0]
        print(f"\nInput: {self.df.loc[recipe_idx, 'title']}")
        
        return self.recommend_by_index(recipe_idx, method=method, top_n=top_n)
    
    def recommend_by_index(self, idx, method='hybrid', top_n=5):
        """
        Recommend recipes by index
        
        Args:
            method: 'tfidf', 'ing_tfidf', 'keyword', 'hybrid'
            top_n: Number of recommendations
        """
        if idx < 0 or idx >= len(self.df):
            print(f"Invalid index: {idx}")
            return None
        
        if method == 'tfidf':
            return get_tfidf_recommendations(idx, self.tfidf_sim, self.df, top_n)
        elif method == 'ing_tfidf':
            return get_ingredient_tfidf_recommendations(idx, self.ing_tfidf_sim, self.df, top_n)
        elif method == 'keyword':
            return get_keyword_recommendations(idx, self.keyword_sim, self.df, top_n)
        elif method == 'hybrid':
            return get_hybrid_recommendations(
                idx, self.hybrid_sim, self.tfidf_sim,
                self.ing_tfidf_sim, self.df, top_n
            )
        else:
            print(f"Unknown method: {method}")
            return None

## 1.6. FoodRecommender Class - Complete Implementation

In [ ]:
# So sánh Ingredient TF-IDF vs Keyword
test_recipe = "canh chua"

print("="*100)
print("INGREDIENT TF-IDF (Better Ingredient Matching)")
print("="*100)
display(recommend_by_title_ingredient_tfidf(test_recipe, data, ingredient_tfidf_similarity, top_n=5))

print("\n" + "="*100)
print("KEYWORD-BASED")
print("="*100)
display(recommend_by_title_keyword(test_recipe, data, keyword_similarity_matrix, top_n=5))

In [ ]:
# Helper functions cho Keyword similarity (sử dụng Jaccard cho keyword sets)
def jaccard_similarity(set1, set2):
    """
    Tính Jaccard similarity giữa 2 sets
    J(A,B) = |A ∩ B| / |A ∪ B|
    """
    if len(set1) == 0 and len(set2) == 0:
        return 0.0
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    return intersection / union if union > 0 else 0.0

def compute_jaccard_similarity_matrix(items_list):
    """
    Tính Jaccard similarity matrix cho list of sets
    Sử dụng cho keyword-based recommendation
    """
    n = len(items_list)
    similarity_matrix = np.zeros((n, n))
    
    for i in range(n):
        for j in range(i, n):
            sim = jaccard_similarity(items_list[i], items_list[j])
            similarity_matrix[i][j] = sim
            similarity_matrix[j][i] = sim
    
    return similarity_matrix

print("Jaccard helper functions defined (for keyword similarity only)!")

In [ ]:
# Hàm extract keywords/tags từ title
def extract_keywords(title):
    """
    Extract main keywords from recipe title
    Focus on: ingredients, cooking methods, dish types
    """
    if pd.isna(title):
        return set()
    
    title = clean_text(title)
    
    # Common Vietnamese cooking methods and dish types
    cooking_methods = ['xào', 'nướng', 'luộc', 'chiên', 'hấp', 'kho', 'rim', 'rang', 
                       'canh', 'súp', 'cháo', 'gỏi', 'nộm', 'salad', 'bún', 'phở', 
                       'mì', 'cơm', 'bánh', 'chè', 'sinh tố']
    
    # Common ingredients
    main_ingredients = ['thịt', 'bò', 'gà', 'heo', 'lợn', 'cá', 'tôm', 'mực', 'nghêu',
                        'rau', 'củ', 'quả', 'trứng', 'đậu', 'nấm', 'măng', 'bí', 
                        'cà', 'khoai', 'su', 'hào', 'cải', 'rau muống', 'rau cần']
    
    # Extract keywords
    keywords = set()
    words = title.split()
    
    # Add cooking methods
    for method in cooking_methods:
        if method in title:
            keywords.add(method)
    
    # Add ingredients (check for 2-word and 1-word matches)
    for i in range(len(words)):
        # Check 2-word combinations
        if i < len(words) - 1:
            two_word = f"{words[i]} {words[i+1]}"
            for ing in main_ingredients:
                if ing in two_word:
                    keywords.add(ing)
        
        # Check single words
        for ing in main_ingredients:
            if ing in words[i]:
                keywords.add(ing)
    
    # Add all significant words (length > 2) as backup
    for word in words:
        if len(word) > 2:
            keywords.add(word)
    
    return keywords

# Apply keyword extraction to all recipes
data['keywords'] = data['title'].apply(extract_keywords)
print(f"Keyword extraction completed!")

# Show examples
print("\nSample keywords:")
for i in range(5):
    print(f"\n{data['title'].iloc[i]}")
    print(f"   Keywords: {data['keywords'].iloc[i]}")

In [ ]:
# Hàm lấy recommendations dựa trên keywords
def get_keyword_recommendations(recipe_idx, similarity_matrix, df, top_n=5):
    sim_scores = list(enumerate(similarity_matrix[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['keyword_score'] = scores
    
    return result

def recommend_by_title_keyword(title, df, similarity_matrix, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\nInput: {df.loc[recipe_idx, 'title']}")
    print(f"   Keywords: {df.loc[recipe_idx, 'keywords']}")
    print("\nKeyword-Based Recommendations:")
    
    return get_keyword_recommendations(recipe_idx, similarity_matrix, df, top_n)

## 1.5. Keyword-Based Recommendation

Extract keywords chính từ title (nguyên liệu chính + phương pháp nấu).

**Ưu điểm**: Đơn giản, focus vào main keywords, dễ giải thích

In [ ]:
# Tính hybrid similarity matrix
hybrid_similarity_matrix = compute_hybrid_similarity(
    tfidf_similarity, 
    ingredient_tfidf_similarity,
    w_tfidf=0.4,
    w_ing_tfidf=0.6
)
print(f"Hybrid Similarity Matrix shape: {hybrid_similarity_matrix.shape}")

In [ ]:
# Hàm tính hybrid similarity (kết hợp TF-IDF và Ingredient TF-IDF)
def compute_hybrid_similarity(tfidf_sim, ing_tfidf_sim, 
                              w_tfidf=0.4, w_ing_tfidf=0.6):
    total_weight = w_tfidf + w_ing_tfidf
    w_tfidf /= total_weight
    w_ing_tfidf /= total_weight
    
    print(f"Weights: TF-IDF={w_tfidf:.2f}, Ingredient TF-IDF={w_ing_tfidf:.2f}")
    
    hybrid_similarity = (w_tfidf * tfidf_sim + 
                        w_ing_tfidf * ing_tfidf_sim)
    
    return hybrid_similarity

## 1.4. Hybrid Approach

Kết hợp 2 phương pháp với trọng số: TF-IDF (0.4) + Ingredient TF-IDF (0.6)

**Rationale**: Ingredients quan trọng nhất trong food recommendation, nên tăng Ingredient TF-IDF weight

In [ ]:
# Test Ingredient TF-IDF Recommendation
test_recipe = "Thịt bò xào"
display(recommend_by_title_ingredient_tfidf(test_recipe, data, ingredient_tfidf_similarity, top_n=10))

In [ ]:
# Chuẩn bị ingredient text cho TF-IDF
# Join ingredients thành string separated by spaces
data['ingredients_text'] = data['ingredients_list'].apply(
    lambda x: ' '.join([clean_text(ing) for ing in x if ing])
)

print("Sample ingredient texts:")
for i in range(3):
    print(f"\n{data['title'].iloc[i]}")
    print(f"   Ingredients text: {data['ingredients_text'].iloc[i][:100]}...")

# Build TF-IDF vectorizer cho ingredients
print("\nBuilding Ingredient TF-IDF matrix...")
ingredient_tfidf_vectorizer = TfidfVectorizer(
    max_features=2000,  # Fewer features than text-based
    ngram_range=(1, 2),  # Unigrams and bigrams
    min_df=2,
    max_df=0.8
)

ingredient_tfidf_matrix = ingredient_tfidf_vectorizer.fit_transform(data['ingredients_text'])
print(f"Ingredient TF-IDF Matrix shape: {ingredient_tfidf_matrix.shape}")

# Tính cosine similarity
ingredient_tfidf_similarity = cosine_similarity(ingredient_tfidf_matrix, ingredient_tfidf_matrix)
print(f"Ingredient TF-IDF Similarity Matrix shape: {ingredient_tfidf_similarity.shape}")

## 1.3. Ingredient TF-IDF Based Recommendation

**Ingredient TF-IDF** xử lý ingredient list như text documents, tốt vì:
- Xử lý được variations trong cách viết (thịt bò, bò, beef...)
- Gán trọng số cho ingredients based on importance
- Không bị ảnh hưởng bởi exact string matching

**Ý tưởng**: Mỗi recipe là 1 document, ingredients là words. Áp dụng TF-IDF để tính similarity.

In [ ]:
# Hàm lấy recommendations dựa trên TF-IDF
def get_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n=5):
    sim_scores = list(enumerate(similarity_matrix[recipe_idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]
    
    recipe_indices = [i[0] for i in sim_scores]
    scores = [i[1] for i in sim_scores]
    
    result = df.iloc[recipe_indices][['title', 'type_of_food', 'calories', 'cook_time']].copy()
    result['tfidf_score'] = scores
    
    return result

def recommend_by_title_tfidf(title, df, similarity_matrix, top_n=5):
    matches = df[df['title'].str.contains(title, case=False, na=False)]
    if len(matches) == 0:
        print(f"No recipe found with title containing: {title}")
        return None
    
    recipe_idx = matches.index[0]
    print(f"\nInput: {df.loc[recipe_idx, 'title']}")
    print(f"   Type: {df.loc[recipe_idx, 'type_of_food']}, Calories: {df.loc[recipe_idx, 'calories']}")
    print("\nTF-IDF Recommendations:")
    
    return get_tfidf_recommendations(recipe_idx, similarity_matrix, df, top_n)

In [ ]:
# Tính cosine similarity matrix từ TF-IDF
tfidf_similarity = cosine_similarity(tfidf_matrix, tfidf_matrix)
print(f"TF-IDF Similarity Matrix shape: {tfidf_similarity.shape}")

## 1.2. TF-IDF Based Recommendation

Tính toán similarity dựa trên nội dung văn bản (title, description, steps) sử dụng TF-IDF và cosine similarity.

In [ ]:
# Kiểm tra kết quả preprocessing
print("Sample ingredients_clean:")
for i, ing in enumerate(data['ingredients_clean'].head(3)):
    print(f"\nRecipe {i+1}: {data['title'].iloc[i]}")
    print(f"Ingredients: {ing}")

In [ ]:
# Áp dụng các hàm preprocessing lên dữ liệu
data['ingredients_list'] = data['ingredients'].apply(parse_list_string)
data['step_list'] = data['step'].apply(parse_list_string)

data['title_clean'] = data['title'].apply(clean_text)
data['description_clean'] = data['description'].apply(clean_text)
data['step_clean'] = data['step_list'].apply(lambda x: ' '.join([clean_text(s) for s in x]))

data['cook_time_minutes'] = data['cook_time'].apply(parse_cook_time)
data['calories_numeric'] = data['calories'].apply(parse_calories)

data['ingredients_clean'] = data['ingredients_list'].apply(
    lambda x: set([clean_text(ing) for ing in x if ing])
)

data[['title', 'title_clean', 'cook_time', 'cook_time_minutes', 'calories', 'calories_numeric']].head()

In [ ]:
# Định nghĩa các hàm xử lý và làm sạch dữ liệu
def parse_list_string(s):
    if pd.isna(s) or s == '[]':
        return []
    try:
        return ast.literal_eval(s)
    except:
        return []

def clean_text(text):
    if pd.isna(text):
        return ''
    text = str(text).lower()
    text = re.sub(r'[^\w\s\u00C0-\u1EF9]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def parse_cook_time(time_str):
    if pd.isna(time_str):
        return np.nan
    time_str = str(time_str).lower()
    minutes = 0
    
    hour_match = re.search(r'(\d+)\s*(?:giờ|h|hour)', time_str)
    if hour_match:
        minutes += int(hour_match.group(1)) * 60
    
    min_match = re.search(r'(\d+)\s*(?:phút|p|min|minute)', time_str)
    if min_match:
        minutes += int(min_match.group(1))
    
    if minutes == 0:
        num_match = re.search(r'(\d+)', time_str)
        if num_match:
            minutes = int(num_match.group(1))
    
    return minutes if minutes > 0 else np.nan

def parse_calories(cal_str):
    if pd.isna(cal_str):
        return np.nan
    cal_str = str(cal_str).replace('.', '').replace(',', '')
    match = re.search(r'(\d+)', cal_str)
    if match:
        return float(match.group(1))
    return np.nan

## 1.1. Data Preprocessing

In [ ]:
# Load dataset và xem thông tin tổng quan
df = pd.read_csv('../../data/all_recipes_final.csv')
print(f"Dataset shape: {df.shape}")
print(f"Phân bố theo nguồn:")
print(df['source'].value_counts())
df.head()

In [ ]:
# Import các thư viện cần thiết
import pandas as pd
import numpy as np
import re
import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity, linear_kernel
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Phần 1: Content-Based Recommendation